# Titanic Dataset EDA and Preprocessing
Author: Randi Sumitro

This notebook performs exploratory data analysis and preprocessing on the Titanic dataset.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

In [ ]:
# Load Titanic dataset
url = "https://web.stanford.edu/class/archive/cs/cs109/cs109.1166/stuff/titanic.csv"
df = pd.read_csv(url)

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

In [ ]:
print("\nDataset Info:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nStatistical Summary:")
print(df.describe())

In [ ]:
# Exploratory Data Analysis
plt.figure(figsize=(15, 10))

# Survival distribution
plt.subplot(2, 3, 1)
sns.countplot(x='Survived', data=df)
plt.title('Survival Distribution')

# Gender distribution
plt.subplot(2, 3, 2)
sns.countplot(x='Sex', data=df)
plt.title('Gender Distribution')

# Age distribution
plt.subplot(2, 3, 3)
sns.histplot(df['Age'].dropna(), bins=30, kde=True)
plt.title('Age Distribution')

# Passenger class distribution
plt.subplot(2, 3, 4)
sns.countplot(x='Pclass', data=df)
plt.title('Passenger Class Distribution')

# Survival by gender
plt.subplot(2, 3, 5)
sns.countplot(x='Sex', hue='Survived', data=df)
plt.title('Survival by Gender')

# Survival by passenger class
plt.subplot(2, 3, 6)
sns.countplot(x='Pclass', hue='Survived', data=df)
plt.title('Survival by Passenger Class')

plt.tight_layout()
plt.savefig('eda_plots.png')
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.savefig('correlation_heatmap.png')
plt.show()

In [ ]:
# Preprocessing
class TitanicPreprocessor:
    def __init__(self):
        self.scaler = StandardScaler()
        self.label_encoder = LabelEncoder()
        self.imputer_num = SimpleImputer(strategy='median')
        self.imputer_cat = SimpleImputer(strategy='most_frequent')
        
    def fit_transform(self, df):
        """Fit preprocessing parameters and transform data"""
        df_processed = df.copy()
        
        # Drop unnecessary columns
        columns_to_drop = ['Name']
        if 'Name' in df_processed.columns:
            df_processed = df_processed.drop(columns=columns_to_drop)
        
        # Handle missing values
        numeric_columns = df_processed.select_dtypes(include=[np.number]).columns
        categorical_columns = df_processed.select_dtypes(include=['object']).columns
        
        if len(numeric_columns) > 0:
            df_processed[numeric_columns] = self.imputer_num.fit_transform(df_processed[numeric_columns])
        
        if len(categorical_columns) > 0:
            df_processed[categorical_columns] = self.imputer_cat.fit_transform(df_processed[categorical_columns])
        
        # Encode categorical variables
        for col in categorical_columns:
            if col in df_processed.columns:
                df_processed[col] = self.label_encoder.fit_transform(df_processed[col])
        
        # Scale numerical features
        if len(numeric_columns) > 0:
            df_processed[numeric_columns] = self.scaler.fit_transform(df_processed[numeric_columns])
        
        return df_processed
    
    def transform(self, df):
        """Transform new data using fitted parameters"""
        df_processed = df.copy()
        
        # Drop unnecessary columns
        columns_to_drop = ['Name']
        if 'Name' in df_processed.columns:
            df_processed = df_processed.drop(columns=columns_to_drop)
        
        # Handle missing values
        numeric_columns = df_processed.select_dtypes(include=[np.number]).columns
        categorical_columns = df_processed.select_dtypes(include=['object']).columns
        
        if len(numeric_columns) > 0:
            df_processed[numeric_columns] = self.imputer_num.transform(df_processed[numeric_columns])
        
        if len(categorical_columns) > 0:
            df_processed[categorical_columns] = self.imputer_cat.transform(df_processed[categorical_columns])
        
        # Encode categorical variables
        for col in categorical_columns:
            if col in df_processed.columns:
                df_processed[col] = self.label_encoder.transform(df_processed[col])
        
        # Scale numerical features
        if len(numeric_columns) > 0:
            df_processed[numeric_columns] = self.scaler.transform(df_processed[numeric_columns])
        
        return df_processed

print('Preprocessor class defined!')

In [ ]:
# Apply preprocessing
preprocessor = TitanicPreprocessor()
df_processed = preprocessor.fit_transform(df)

print("\nProcessed Dataset Shape:", df_processed.shape)
print("\nFirst 5 rows of processed data:")
print(df_processed.head())

In [ ]:
print("\nProcessed Dataset Info:")
df_processed.info()

In [ ]:
# Save processed data
import os
os.makedirs('data_preprocessed', exist_ok=True)

df_processed.to_csv('data_preprocessed/titanic_processed.csv', index=False)
print("\nProcessed data saved to 'data_preprocessed/titanic_processed.csv'")

# Save preprocessing parameters
import pickle
with open('data_preprocessed/preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)
print("Preprocessor saved to 'data_preprocessed/preprocessor.pkl'")